Kiểm tra Java & Cài đặt Hadoop 3.5.0

In [1]:
!java -version

[0.025s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.025s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
openjdk version "21.0.12.1" 2026-08-18
OpenJDK Runtime Environment (build 21.0.12.1+1-1-24.04.4-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12.1+1-1-24.04.4-Ubuntu, mixed mode, sharing)


In [2]:
!wget -q --show-progress https://archive.apache.org/dist/hadoop/common/hadoop-3.5.0/hadoop-3.5.0.tar.gz

hadoop-3.5.0.tar.gz 100%[===================>] 554.35M  81.6KB/s    in 54m 21s 


In [3]:
!tar -xzf hadoop-3.5.0.tar.gz

Thiết lập Biến môi trường

In [5]:
import os
import subprocess
os.environ["HADOOP_HOME"] = "/content/hadoop-3.5.0"
os.environ["JAVA_HOME"] = os.path.realpath(
    subprocess.check_output(
        ["bash", "-lc", "dirname $(dirname $(readlink -f $(which java)))"],
        text=True
    ).strip()
)
os.environ["PATH"] = os.environ["HADOOP_HOME"] + "/bin:" + os.environ["PATH"]
with open(os.environ["HADOOP_HOME"] + "/etc/hadoop/hadoop-env.sh", "a") as f:
    f.write("\nexport JAVA_HOME=" + os.environ["JAVA_HOME"] + "\n")

In [6]:
!hadoop version

[0.051s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.051s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Hadoop 3.5.0
Source code repository https://github.com/apache/hadoop.git -r dbcc7cd797100e6b32cd84f85b53a5193a5f9af0
Compiled by cnauroth on 2026-03-24T16:56Z
Compiled on platform linux-x86_64
Compiled with protoc 3.25.5
From source with checksum 64582ed0a62a169c39a19c734a8f33b5
This command was run using /content/hadoop-3.5.0/share/hadoop/common/hadoop-common-3.5.0.jar


In [18]:
from google.colab import files

# Chạy dòng này sẽ hiển thị nút "Choose Files" / "Chọn tệp"
uploaded = files.upload()

Saving orders_2026_09.csv to orders_2026_09.csv


Cấu hình và Khởi động HDFS Pseudo-Distributed

1 Ghi cấu hình XML

In [14]:
%%writefile /content/hadoop-3.5.0/etc/hadoop/core-site.xml
<configuration>
    <property>
        <name>fs.defaultFS</name>
        <value>hdfs://localhost:9000</value>
    </property>
</configuration>

Overwriting /content/hadoop-3.5.0/etc/hadoop/core-site.xml


In [15]:
%%writefile /content/hadoop-3.5.0/etc/hadoop/hdfs-site.xml
<configuration>
    <property>
        <name>dfs.replication</name>
        <value>1</value>
    </property>
    <property>
        <name>dfs.namenode.name.dir</name>
        <value>file:///content/hadoop_data/namenode</value>
    </property>
    <property>
        <name>dfs.datanode.data.dir</name>
        <value>file:///content/hadoop_data/datanode</value>
    </property>
</configuration>

Overwriting /content/hadoop-3.5.0/etc/hadoop/hdfs-site.xml


2 Format NameNode & Khởi động Daemons

In [16]:
!mkdir -p /content/hadoop_data/namenode /content/hadoop_data/datanode
!hdfs namenode -format -force
!hdfs --daemon start namenode
!hdfs --daemon start datanode
!jps

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
2026-10-14 07:51:17,290 INFO namenode.NameNode: STARTUP_MSG: 
/************************************************************
STARTUP_MSG: Starting NameNode
STARTUP_MSG:   host = e603d58cf37d/172.28.0.12
STARTUP_MSG:   args = [-format, -force]
STARTUP_MSG:   version = 3.5.0
STARTUP_MSG:   classpath = /content/hadoop-3.5.0/etc/hadoop:/content/hadoop-3.5.0/share/hadoop/common/lib/hadoop-gcp-3.5.0.jar:/content/hadoop-3.5.0/share/hadoop/common/lib/commons-text-1.14.0.jar:/content/hadoop-3.5.0/share/hadoop/common/lib/netty-transport-4.1.130.Final.jar:/content/hadoop-3.5.0/share/hadoop/common/lib/jsr305-3.0.2.jar:/content/hadoop-3.5.0/share/hadoop/common/lib/jakarta.anno

Đưa dữ liệu orders_2026_09.csv lên HDFS

In [19]:
!hdfs dfs -mkdir -p /user/student/ecommerce/input
!hdfs dfs -mkdir -p /user/student/ecommerce/output
!hdfs dfs -put -f /content/orders_2026_09.csv /user/student/ecommerce/input/

# Kiểm tra dữ liệu (Lấy minh chứng nộp bài)
!hdfs dfs -ls /user/student/ecommerce/input
!hdfs dfs -head /user/student/ecommerce/input/orders_2026_09.csv

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warn

Viết mapper_revenue.py

In [20]:
%%writefile mapper_revenue.py
import sys
import csv

# Đọc từng dòng CSV qua DictReader
reader = csv.DictReader(sys.stdin)
for row in reader:
    # Chỉ lấy các đơn hàng thành công SUCCESS
    if row["status"].strip() == "SUCCESS":
        category = row["category"].strip()
        amount = int(row["amount"].strip())
        # Phát sinh cặp Key-Value: category \t amount
        print(f"{category}\t{amount}")

Writing mapper_revenue.py


Viết reducer_revenue.py

In [21]:
%%writefile reducer_revenue.py
import sys

current_category = None
current_sum = 0

for line in sys.stdin:
    line = line.strip()
    if not line:
        continue
    category, amount = line.split("\t")
    amount = int(amount)

    if current_category == category:
        current_sum += amount
    else:
        if current_category is not None:
            print(f"{current_category}\t{current_sum}")
        current_category = category
        current_sum = amount

if current_category is not None:
    print(f"{current_category}\t{current_sum}")

Writing reducer_revenue.py


Cấp quyền thực thi và test thử bằng pipeline trên Linux:

In [22]:
!chmod +x mapper_revenue.py reducer_revenue.py
!cat /content/orders_2026_09.csv | python3 mapper_revenue.py | sort | python3 reducer_revenue.py

Beauty	111032003
Book	39306022
Electronics	3703769311
Fashion	218597763
Grocery	80983147
Home	410315065
Sport	166194354
Toy	83881779


Chạy MapReduce qua Hadoop Streaming

In [23]:
!hdfs dfs -rm -r -f /user/student/ecommerce/output/revenue_by_category

!hadoop jar $HADOOP_HOME/share/hadoop/tools/lib/hadoop-streaming*.jar \
  -files mapper_revenue.py,reducer_revenue.py \
  -mapper "python3 mapper_revenue.py" \
  -reducer "python3 reducer_revenue.py" \
  -input /user/student/ecommerce/input/orders_2026_09.csv \
  -output /user/student/ecommerce/output/revenue_by_category

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
2026-10-14 07:56:11,118 WARN impl.MetricsSystemImpl: JobTracker metrics system already initialized!
2026-10-14 07:56:11,638 INFO mapred.FileInputFormat: Total input files to process : 1
2026-10-14 07:56:11,734 INFO mapreduce.JobSubmitter: number of splits:1
2026-10-14 07:56:12,096 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_

Đọc file kết quả part-00000

In [24]:
!hdfs dfs -cat /user/student/ecommerce/output/revenue_by_category/part-*

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Beauty	111032003
Book	39306022
Electronics	3703769311
Fashion	218597763
Grocery	80983147
Home	410315065
Sport	166194354
Toy	83881779


Sắp xếp lấy Top 3 nhóm có doanh thu cao nhất

In [25]:
!hdfs dfs -cat /user/student/ecommerce/output/revenue_by_category/part-* | sort -k2,2nr | head -3

Electronics	3703769311
Home	410315065
Fashion	218597763
